# Train Plant Disease Detection — Google Colab

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**

**Trên máy local:** nén `ml/data/split/` → upload `split.zip` lên Google Drive.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cấu hình đường dẫn

**Trên Drive cần có:**
- `Project_plant_disease/data/split.zip`
- `Project_plant_disease/ml_code.zip` — nén từ máy: folder `ml/src` + `ml/configs` + `ml/requirements.txt`

**Không dùng Git** nếu repo chưa push GitHub → để `USE_GIT = False`.

In [ ]:
from pathlib import Path

# === Drive paths ===
DRIVE_SPLIT_ZIP = "/content/drive/MyDrive/Project_plant_disease/data/split.zip"
DRIVE_ML_ZIP = "/content/drive/MyDrive/Project_plant_disease/ml.zip"

# False = dùng ml.zip trên Drive
USE_GIT = False
REPO_URL = "https://github.com/YOUR_TEAM/plant-disease-detection.git"

WORK = Path("/content/plant-disease-detection/ml")

for p, name in [(DRIVE_SPLIT_ZIP, "split.zip"), (None if USE_GIT else DRIVE_ML_ZIP, "ml_code.zip")]:
    if p and not Path(p).exists():
        raise FileNotFoundError(f"Không thấy {name}: {p}")

print("split.zip OK")
if not USE_GIT:
    print("ml.zip OK")

In [ ]:
import shutil
import zipfile
from pathlib import Path

REPO = Path("/content/plant-disease-detection")
WORK.mkdir(parents=True, exist_ok=True)

# --- Lấy code train ---
if USE_GIT:
    train_py = WORK / "src" / "train.py"
    if not train_py.exists():
        if REPO.exists():
            shutil.rmtree(REPO)
        !git clone {REPO_URL} {REPO}
else:
    # Giải nén ml.zip → /content/plant-disease-detection/ml/
    !unzip -qo "{DRIVE_ML_ZIP}" -d /content/plant-disease-detection/
    # Hỗ trợ zip có cấu trúc ml/src hoặc src/ ngay trong zip
    if not (WORK / "src" / "train.py").exists() and (REPO / "src" / "train.py").exists():
        for item in ["src", "configs", "requirements.txt"]:
            src = REPO / item
            if src.exists():
                dest = WORK / item
                if dest.exists() and dest.is_dir():
                    shutil.rmtree(dest)
                shutil.move(str(src), str(WORK / item))

assert (WORK / "src" / "train.py").exists(), (
    "Không thấy src/train.py.\n"
    "→ Upload ml_code.zip (src + configs + requirements.txt) lên Drive,\n"
    "   hoặc sửa REPO_URL nếu dùng Git."
)
print("Code OK:", WORK / "src" / "train.py")

# --- Giải nén split.zip ---
data_dir = WORK / "data"
data_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(DRIVE_SPLIT_ZIP) as zf:
    zf.extractall(data_dir)

# split.zip nén từ data/split → giải ra data/split/
# hoặc nén trực tiếp train/val/test → giải ra data/train/
if not (data_dir / "split" / "train").exists():
    if (data_dir / "split" / "split" / "train").exists():
        shutil.move(str(data_dir / "split" / "split"), str(data_dir / "split_tmp"))
        shutil.rmtree(data_dir / "split")
        shutil.move(str(data_dir / "split_tmp"), str(data_dir / "split"))
    elif (data_dir / "train").exists():
        (data_dir / "split").mkdir(exist_ok=True)
        for subset in ("train", "val", "test"):
            src = data_dir / subset
            if src.exists():
                shutil.move(str(src), str(data_dir / "split" / subset))

assert (data_dir / "split" / "train").exists(), "Không thấy data/split/train — kiểm tra cách nén split.zip"
print("Data OK:", data_dir / "split" / "train")

%cd {WORK}

In [ ]:
!pip install -q tqdm pyyaml scikit-learn matplotlib seaborn Pillow

In [ ]:
!python src/train.py

In [ ]:
# Copy model về Drive
OUT = "/content/drive/MyDrive/Project_plant_disease/models"
import shutil
from pathlib import Path
Path(OUT).mkdir(parents=True, exist_ok=True)
shutil.copytree("models", OUT, dirs_exist_ok=True)
print("Đã lưu:", OUT)
print("- best_model.pt")
print("- classes.json")